## **Climate-Based Crop Yield Estimation Using Artificial Neural Network**

Import necessary libraries

In [ ]:
# Data manipulation and numerical operations
import pandas as pd
import numpy as np

# Visualization
import seaborn as sns
import matplotlib.pyplot as plt

# Data preprocessing and splitting
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# TensorFlow & Keras for building the machine learning model
import tensorflow as tf
from tensorflow.keras import regularizers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Evaluation metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

### **The Dataset**

*The “Climate Change Impact on Agriculture” dataset obtained from Kaggle, contains 10,000 samples that covers various climate and agricultural factors. It is a synthetic dataset designed to simulate real-world agricultural data, which will be used to explore how climate variations influence food production and then develop the predictive model.*

Loading the Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('climate_change_impact_on_agriculture.csv')


Data Cleaning and Preprocessing

In [ ]:
# Drop rows with missing values
df = df.dropna()

# Separate features and target
X = df[['Average_Temperature_C', 'Total_Precipitation_mm', 'CO2_Emissions_MT', 'Extreme_Weather_Events']]
y = df['Crop_Yield_MT_per_HA']

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


### **Exploratory Data Analysis**

In [ ]:
# Correlation matrix heatmap
plt.figure(figsize=(8,6))
correlation_matrix = df[['Average_Temperature_C', 'Total_Precipitation_mm', 'CO2_Emissions_MT', 'Extreme_Weather_Events']].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix of Climate-Related Features')
plt.show()

In [ ]:
# Histograms for the climate related features
histogram = df[['Average_Temperature_C', 'Total_Precipitation_mm', 'CO2_Emissions_MT', 'Extreme_Weather_Events']]
histogram.hist(figsize=(10,8), bins=20)
plt.suptitle('Climate-Related Feature Distributions')
plt.show()

In [ ]:
# Pairplot relationships between climate-related features and crop yield
climate_features = ['Average_Temperature_C', 'Total_Precipitation_mm', 'CO2_Emissions_MT', 'Extreme_Weather_Events', 'Crop_Yield_MT_per_HA']
sns.pairplot(df[climate_features])
plt.suptitle('Relationships Between Climate-Related Features and Crop Yield', y=1.02)
plt.show()

### **Model Development**

 Model Architecture

In [ ]:
# Define a feedforward neural network for regression
model = Sequential()
model.add(Input(shape=(X_scaled.shape[1],)))   # Input layer

model.add(Dense(128, activation='relu'))        # Hidden layer 1
model.add(Dropout(0.4))                        # Dropout layer

model.add(Dense(64, activation='relu'))        # Hidden layer 2
model.add(Dropout(0.3))                        # Dropout layer

model.add(Dense(32, activation='relu'))        # Hidden layer 3

model.add(Dense(1, activation='linear'))       # Output layer


### **Training and Evaluation**

Train-Test Split

In [ ]:
# Split dataset into train and test
X_train_full, X_test, y_train_full, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Further split train into train and validation sets 
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=42)


Compile and Train the Model

In [ ]:
# Compile the model with optimizer, loss, and evaluation metric
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])

# # Set up early stopping to avoid overfitting
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Train the model with validation data
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping]
)

Evaluate Model Performance on Test Set

In [ ]:
# Predict crop yield on the test set
y_pred = model.predict(X_test).flatten()

# Compute evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

# Print the results
print(f'Test MAE: {mae}')
print(f'Test RMSE: {rmse:.4f}')
print(f'Test R² Score: {r2:.4f}')

Visualize Training and Validation Loss

In [ ]:
# Plot training & validation loss values over epochs
plt.figure(figsize=(8,5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss During Training')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

Predict Crop Yield from New Climate Data

In [ ]:
# Example climate values for prediction 
new_climate = pd.DataFrame({
    'Average_Temperature_C': [26.5],
    'Total_Precipitation_mm': [920],
    'CO2_Emissions_MT': [3.1],
    'Extreme_Weather_Events': [2]
})

# Scale the new input using the same scaler as training data
new_climate_scaled = scaler.transform(new_climate)

# Predict crop yield using the trained model
predicted_yield = model.predict(new_climate_scaled)

# Display the predicted yield
print("Estimated Crop Yield:", predicted_yield[0][0], "MT/HA")


### **Bonus: Model Benchmarking and Comparison**

In [ ]:
# Compare with baseline ML models
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(random_state=42),
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    maes = mean_absolute_error(y_test, preds)
    rmses = np.sqrt(mean_squared_error(y_test, preds))
    r2s = r2_score(y_test, preds)
    results.append({'Model': name, 'MAE': maes, 'RMSE': rmses, 'R²': r2s})

# Include ANN results for comparison
results.append({'Model': 'ANN', 'MAE': mae, 'RMSE': rmse, 'R²': r2})

# Display results
results_df = pd.DataFrame(results)
print(results_df.sort_values(by='MAE'))


*Plot Model Comparison*

The comparison of regression models shows that the Artificial Neural Network (ANN) outperformed both the Random Forest and Linear Regression models in predicting crop yield based on climate-related features. The ANN achieved the lowest mean absolute error (0.7285) and root mean squared error (0.8594), indicating more accurate predictions, along with the highest R² score (0.3003), meaning it explained about 30% of the variation in crop yield. While this suggests the ANN captured more relevant patterns than the other models, the moderate R² also highlights the potential to further improve prediction accuracy.